# Toric-code Hamiltonian graph colored by an eigenstate

This notebook does the following:

1. Build a small periodic toric-code Hamiltonian with `qlinks`.
2. Diagonalize the sparse Hamiltonian.
3. Select one eigenstate.
4. Draw the Fock-space / Hamiltonian graph colored by that eigenstate's amplitudes.

The default system is `2×2` with periodic boundary conditions, so the Hilbert-space dimension is `2**8 = 256`.

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh

# Direct imports are robust even if package-level __init__ exports are stale.
from qlinks.models.toric_code import ToricCodeModel
from qlinks.visualizer.hamiltonian_graph import (
    HamiltonianGraphStyle,
    HamiltonianGraphVisualizer,
)


## 1. Build the toric-code Hamiltonian

For the current implementation, `ToricCodeModel` supports periodic boundary conditions and the sparse builder.

In [ ]:
model = ToricCodeModel(
    lx=2,
    ly=2,
    boundary_condition="periodic",
    electric=1.0,
    magnetic=1.0,
)

# Prefer build() over build_hamiltonian() when you also want access to the basis,
# kinetic term, and potential term.
result = model.build(
    builder="sparse",
    basis_solver="dfs",
    sort_basis=True,
    on_missing="raise",
)

H = result.hamiltonian.tocsr()
K = result.kinetic.tocsr() if result.kinetic is not None else None
V = result.potential.tocsr() if result.potential is not None else None
basis = result.basis

print(f"num links        = {model.lattice.num_links}")
print(f"basis dimension  = {basis.n_states}")
print(f"H shape          = {H.shape}")
print(f"H.nnz            = {H.nnz}")
print(f"K.nnz            = {None if K is None else K.nnz}")
print(f"V.nnz            = {None if V is None else V.nnz}")


## 2. Diagonalize

For the `2×2` system, sparse diagonalization with `eigsh` is enough. The toric code has degeneracies, so individual eigenvectors inside a degenerate subspace are not unique. For visualization, that is fine: choose one returned eigenvector, or later construct your own superposition inside the degenerate subspace.

In [ ]:
num_eigs = 8
evals, evecs = eigsh(H, k=num_eigs, which="SA")

order = np.argsort(evals)
evals = evals[order]
evecs = evecs[:, order]

print("Lowest eigenvalues:")
for index, energy in enumerate(evals):
    print(f"{index:2d}: E = {energy:.12g}")


## 3. Select one eigenstate

The graph coloring can use `state_weight`, `state_amplitude_real`, `state_amplitude_imag`, `state_amplitude_abs`, or `state_phase`.

In [ ]:
state_index = 1
psi = evecs[:, state_index]
energy = evals[state_index]

# Normalize defensively.
psi = psi / np.linalg.norm(psi)

print(f"Selected eigenstate index = {state_index}")
print(f"Selected energy           = {energy:.12g}")
print(f"Norm                      = {np.linalg.norm(psi):.12g}")
print(f"Max |amplitude|           = {np.max(np.abs(psi)):.12g}")


## 4. Draw the Hamiltonian graph colored by the eigenstate

Usually it is cleaner to draw the graph from the kinetic/off-diagonal term `K`, because the potential term only contributes diagonal self-loop values. If you want graph edges from every off-diagonal term of the full Hamiltonian, using `H` is also fine because the visualizer removes diagonal entries by default.

In [ ]:
graph_matrix = K if K is not None else H

style = HamiltonianGraphStyle(
    vertex_size=10.0,
    edge_width=0.4,
    edge_alpha=0.25,
    label_vertices=False,
    colorbar=True,
    cmap="viridis",
    figure_size=(8.0, 8.0),
)

visualizer = HamiltonianGraphVisualizer.from_sparse_matrix(
    graph_matrix,
    include_self_loops=False,
    weight_tolerance=1e-12,
    style=style,
)

# Use backend="networkx" to avoid requiring python-igraph.
visualizer.plot(
    backend="igraph",
    layout="mds",
    color_by="state_amplitude_real",
    state_vector=psi,
    title=f"Toric code {model.lx}x{model.ly}, eigenstate {state_index}, E={energy:.6g}",
)


## Optional: compare different amplitude colorings

In [ ]:
for color_by in ["state_amplitude_real", "state_amplitude_abs", "state_phase"]:
    visualizer.plot(
        backend="networkx",
        layout="spring",
        color_by=color_by,
        state_vector=psi,
        title=f"{color_by}, eigenstate {state_index}, E={energy:.6g}",
    )


## Optional: inspect the basis configurations with the largest weights

This is useful because a Fock-space node is a basis configuration. The exact attribute for accessing the raw basis array may change, so this cell tries a few common names.

In [ ]:
def basis_array_from_basis(basis):
    for attr in ["configs", "states", "basis", "array", "data"]:
        if hasattr(basis, attr):
            value = getattr(basis, attr)
            if isinstance(value, np.ndarray):
                return value
    if hasattr(basis, "to_numpy"):
        return basis.to_numpy()
    if hasattr(basis, "as_array"):
        return basis.as_array()
    return None

basis_array = basis_array_from_basis(basis)
weights = np.abs(psi) ** 2
top = np.argsort(weights)[::-1][:10]

for node in top:
    print(f"node={node:4d}  weight={weights[node]:.6g}  amp={psi[node]:+.6g}")
    if basis_array is not None:
        print("    config =", basis_array[node])
